# Audio Classification from Embeddings

This notebook created by [biodiversica](https://biodiversica.xyz) classifies audio recordings by running a **classifier head model** over pre-computed embeddings stored in a SQLite database.

This is the second step of a two-stage pipeline:
1. **Compute Embeddings Database** — extract embeddings from a foundation model backbone
2. **This notebook** — apply a lightweight classifier head on top of those embeddings

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read the embeddings database and save results
2. **Installs the necessary software** automatically
3. **Loads** the classifier head model, labels file, and database metadata
4. **Checks** which recordings have already been classified
5. **Runs classification** on each audio segment and saves results to Google Drive

### Classifier head model:
The model must accept an embedding vector as input and output raw logits (one value per class).
Supported formats: **ONNX** (`.onnx`) and **TFLite** (`.tflite`).
The model can be loaded from Google Drive or downloaded from HuggingFace.

### Result files:
Results are saved in the same format as the audio analyzer notebooks —
one `.results.txt` file per recording, under `RESULTS_FOLDER / SITE_NAME /`.
Columns: `site`, `lat`, `lon`, `date`, `time`, `start_time`, `end_time`, `label`, `confidence`.

### How to run:
Run each cell **one at a time**, top to bottom. Click ▶ or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` has not been run. After running it shows `[1]`, `[2]`, etc.

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive**.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!pip install ai-edge-litert onnxruntime huggingface_hub -q

print('\nAll packages installed successfully.')

---
## Step 2 — Configuration

Fill in the four forms below and run them top to bottom.

1. **General Settings** — results folder, confidence threshold, top-K, site coordinates
2. **Embeddings Database** — path to your `.db` file and optional filename prefix
3. **Classifier Head Model** — model file and inference settings
4. **Labels** — labels file and parsing settings

> **Tip:** The forms hide the code automatically. Click `{ }` in the top-right corner of any form cell to see the underlying code.

In [ ]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Results folder** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/classification_results'  #@param {type:"string"}

#@markdown ---
#@markdown **Detection threshold** — minimum confidence score to keep a detection (0.0–1.0).
#@markdown Lower = more detections (may include false positives). Higher = fewer but more reliable.
MIN_CONFIDENCE = 0.25  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Top K detections per segment** — maximum detections to keep per audio segment.
#@markdown 1 = only the highest-confidence detection. Increase to keep more.
TOP_K = 1  #@param {type:"integer"}

#@markdown ---
#@markdown **Site coordinates** (optional) — latitude and longitude per recording site.
#@markdown Format: `SITE_NAME=lat,lon` separated by semicolons.
#@markdown Example: `POINT_A=-15.1,-47.2;POINT_B=-16.3,-48.1`
SITE_COORDINATES = ''  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try:
            _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception:
            pass

print(f'Results folder  : {DRIVE_RESULTS_DIR}')
print(f'Min. confidence : {MIN_CONFIDENCE}')
print(f'Top K           : {TOP_K}')
if _latlon_map:
    for _sn, (_la, _lo) in _latlon_map.items():
        print(f'  Coordinates   : {_sn} → lat={_la}, lon={_lo}')

In [ ]:
#@title 🗄️ Embeddings Database { display-mode: "form" }

import os

#@markdown **Database path** — full path to the `.db` file created by the *Compute Embeddings Database* notebook.
EMBEDDINGS_DB_PATH = '/content/drive/MyDrive/embeddings/my_model.embeddings.db'  #@param {type:"string"}

#@markdown ---
#@markdown **Filename prefix** — used to parse recording timestamps from the embedding keys.
#@markdown Leave blank for standard AudioMoth filenames (`YYYYMMDD_HHMMSS.wav`).
#@markdown Enter `SM4` if files were named like `SM4_20250615_203000.wav`.
FILENAME_PREFIX = ''  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()

if not os.path.exists(EMBEDDINGS_DB_PATH):
    print(f'WARNING: Database not found: {EMBEDDINGS_DB_PATH}')
    print('Check the path above — make sure Google Drive is connected.')
else:
    import sqlite3
    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _c:
        _n = _c.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
    print(f'Database : {EMBEDDINGS_DB_PATH}')
    print(f'Embeddings stored : {_n}')

In [ ]:
#@title 🤖 Classifier Head Model { display-mode: "form" }

#@markdown **Model source** — where the classifier head model file is stored.
MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown ---
#@markdown ### If model source = Google Drive
#@markdown Full path to the model file on your Drive (`.onnx` or `.tflite`).
DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/classifier_head.onnx'  #@param {type:"string"}

#@markdown ---
#@markdown ### If model source = HuggingFace
HF_REPO_ID    = ''  #@param {type:"string"}
HF_MODEL_FILE = 'classifier_head.onnx'  #@param {type:"string"}

#@markdown ---
#@markdown **Activation function** — how the model converts raw outputs to confidence scores.
#@markdown `sigmoid` = each class independently (most common). `softmax` = scores sum to 1. `none` = model already outputs probabilities.
ACTIVATION = 'sigmoid'  #@param ["sigmoid", "softmax", "none"]

#@markdown ---
#@markdown ### ONNX settings *(ignored for TFLite models)*
#@markdown **Input node name** — name of the ONNX input tensor (the embedding vector).
ONNX_INPUT_NAME = 'embedding'  #@param {type:"string"}
#@markdown **Output node name** — name of the ONNX output tensor (the logits).
ONNX_OUTPUT_NAME = 'logits'  #@param {type:"string"}

print(f'Model source : {MODEL_SOURCE}')
print(f'Activation   : {ACTIVATION}')
if MODEL_SOURCE == 'google_drive':
    print(f'Model path   : {DRIVE_MODEL_PATH}')
else:
    print(f'HF repo      : {HF_REPO_ID}')
    print(f'HF file      : {HF_MODEL_FILE}')
if ONNX_INPUT_NAME or ONNX_OUTPUT_NAME:
    print(f'ONNX input   : {ONNX_INPUT_NAME}')
    print(f'ONNX output  : {ONNX_OUTPUT_NAME}')

In [ ]:
#@title 🏷️ Labels { display-mode: "form" }

#@markdown **Labels source** — where the labels file is stored.
LABELS_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown ---
#@markdown ### If labels source = Google Drive
#@markdown Full path to the labels file on your Drive (plain text, one label per line).
DRIVE_LABELS_PATH = '/content/drive/MyDrive/models/labels.txt'  #@param {type:"string"}

#@markdown ---
#@markdown ### If labels source = HuggingFace
#@markdown Leave **HF labels repo** blank to use the same repo as the model.
HF_LABELS_REPO = ''  #@param {type:"string"}
HF_LABELS_FILE = 'labels.txt'  #@param {type:"string"}

#@markdown ---
#@markdown ### Labels file settings
#@markdown **Has header row?** — check if the first line is a column header, not a label.
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Label column index** — which column contains the label (0 = first column).
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Column delimiter** — how columns are separated in the labels file.
LABELS_DELIMITER = 'underscore (_)'  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

print(f'Labels source  : {LABELS_SOURCE}')
if LABELS_SOURCE == 'google_drive':
    print(f'Labels path    : {DRIVE_LABELS_PATH}')
else:
    print(f'HF labels repo : {HF_LABELS_REPO or "(same as model repo)"}')
    print(f'HF labels file : {HF_LABELS_FILE}')
print(f'Has header     : {LABELS_HAS_HEADER}')
print(f'Column index   : {LABELS_COLUMN_INDEX}')
print(f'Delimiter      : {LABELS_DELIMITER}')

---
## Step 3 — Load Model, Labels & Database Metadata

This cell downloads the classifier head (if from HuggingFace), loads the model and labels file,
and reads the embedding parameters stored in the database (window size, embedding size, sample rate).

In [ ]:
#@title 🧠 Load model, labels & database metadata { display-mode: "form" }

import os
import sqlite3
import numpy as np

# --- Load classifier head model ---
if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Downloading model from HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
else:
    model_path = DRIVE_MODEL_PATH

if not os.path.exists(model_path):
    raise FileNotFoundError(f'Model not found: {model_path}\nCheck your model configuration in Step 2.')

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'. File must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    _clf_model = TFLiteInterpreter(model_path=model_path)
    _clf_model.allocate_tensors()
    in_shape  = _clf_model.get_input_details()[0]['shape']
    out_shape = _clf_model.get_output_details()[0]['shape']
    print('TFLite classifier loaded.')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort
    _clf_model = ort.InferenceSession(model_path)
    in_shape  = _clf_model.get_inputs()[0].shape
    out_shape = _clf_model.get_outputs()[0].shape
    print('ONNX classifier loaded.')
print(f'  Path         : {model_path}')
print(f'  Input  shape : {in_shape}')
print(f'  Output shape : {out_shape}')
if MODEL_TYPE == 'onnx':
    print(f'  Providers    : {_clf_model.get_providers()}')

# --- Load labels ---
_DELIMITERS = {'tab': '\t', 'comma (,)': ',', 'semicolon (;)': ';', 'underscore (_)': '_'}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if LABELS_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    _lrepo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'\nDownloading labels from HuggingFace: {_lrepo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_lrepo, filename=HF_LABELS_FILE)
else:
    labels_path = DRIVE_LABELS_PATH

if not os.path.exists(labels_path):
    raise FileNotFoundError(f'Labels file not found: {labels_path}')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    _lines = f.readlines()
if LABELS_HAS_HEADER and _lines:
    _lines = _lines[1:]
for _line in _lines:
    _line = _line.strip()
    if not _line:
        continue
    if _labels_sep in _line:
        _parts = _line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(_parts):
            labels.append(_parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f'  WARNING: column index {LABELS_COLUMN_INDEX} not found in: {_line!r}')
    else:
        labels.append(_line)

print(f'\nLabels loaded  : {len(labels)} classes')
print(f'  First 5      : {labels[:5]}')
print(f'  Model name   : {MODEL_NAME}')

# --- Read database metadata ---
if not os.path.exists(EMBEDDINGS_DB_PATH):
    raise FileNotFoundError(f'Embeddings database not found: {EMBEDDINGS_DB_PATH}')

with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
    _db_meta = dict(_con.execute('SELECT key, value FROM metadata').fetchall())

# Support both the new auricularia format and the legacy notebook format key names.
DB_WINDOW_S       = float(_db_meta.get("window_duration_s", _db_meta.get("window_size_s", 5.0)))
DB_EMBEDDING_SIZE = int(_db_meta.get("embedding_size", 1536))
DB_SAMPLE_RATE    = int(_db_meta.get("sample_rate", 32000))
DB_OVERLAP        = float(_db_meta.get("overlap", 0.0))
DB_MODEL_NAME     = _db_meta.get("model_id", _db_meta.get("model_name", "unknown"))

# --- Load per-site metadata (stream_id, UTC offset, coordinates) ---
_sites_meta = {}
try:
    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _sc:
        for _row in _sc.execute('SELECT site_name, stream_id, utc_offset, lat, lon FROM sites').fetchall():
            _sites_meta[_row[0]] = {'stream_id': _row[1], 'utc_offset': float(_row[2]),
                                    'lat': _row[3], 'lon': _row[4]}
except Exception:
    pass

print('\nEmbeddings database:')
print(f'  Backbone model : {DB_MODEL_NAME}')
print(f'  Sample rate    : {DB_SAMPLE_RATE} Hz')
print(f'  Window size    : {DB_WINDOW_S} s')
print(f'  Embedding size : {DB_EMBEDDING_SIZE}')
print(f'  Overlap        : {DB_OVERLAP}')
if _sites_meta:
    print(f'  Sites          : {len(_sites_meta)}')
    for _sn, _sm in sorted(_sites_meta.items()):
        _utc = _sm['utc_offset']
        _sid = _sm['stream_id']
        print(f"    {_sn}: stream_id='{_sid}', UTC{_utc:+.1f}")

---
## Step 4 — Check Already Completed Analyses

This cell scans the embeddings database and your results folder to see which recordings
have **already been classified**. Those will be **skipped** in Step 5, so you can safely re-run
without repeating work you've already done.

In [ ]:
#@title 🔎 Check already completed analyses { display-mode: "form" }

import os
import re
import sqlite3
from datetime import datetime
from collections import defaultdict

def _parse_stem_datetime(stem, prefix=""):
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})", stem)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf"{re.escape(prefix)}_(\d{{8}})_(\d{{6}})", stem)
    else:
        m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

from datetime import timedelta as _td

def _get_result_path(site, stem):
    _utc_dt  = _parse_stem_datetime(stem, FILENAME_PREFIX)
    _utc_off = _sites_meta.get(site, {}).get('utc_offset', 0.0)
    dt       = _utc_dt + _td(hours=_utc_off) if _utc_dt else None
    if dt is not None:
        result_fn = f"{site}_{dt.strftime('%Y%m%d_%H%M%S')}.{MODEL_NAME}.results.txt"
    else:
        result_fn = f'{site}_{stem}.{MODEL_NAME}.results.txt'
    return os.path.join(DRIVE_RESULTS_DIR, site, result_fn)

def _detect_db_format(con):
    emb_cols = {row[1] for row in con.execute("PRAGMA table_info(embeddings)").fetchall()}
    if "chunk_index" in emb_cols and "site_name" in emb_cols:
        return "new"
    return "notebook"

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# Enumerate unique site/recording pairs from the DB.
_file_keys = []
with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
    DB_FORMAT = _detect_db_format(_con)
    if DB_FORMAT == "new":
        for (site_name, recording_id) in _con.execute(
            'SELECT DISTINCT site_name, recording_id FROM embeddings ORDER BY site_name, recording_id'
        ).fetchall():
            _file_keys.append(f"{site_name}/{recording_id}")
    else:
        _seen = set()
        for (key,) in _con.execute('SELECT key FROM embeddings ORDER BY key'):
            prefix = '/'.join(key.split('/')[:2])
            if prefix not in _seen:
                _seen.add(prefix)
                _file_keys.append(prefix)

to_process   = []
already_done = []
for fk in _file_keys:
    site, stem = fk.split('/', 1)
    if os.path.exists(_get_result_path(site, stem)):
        already_done.append(fk)
    else:
        to_process.append(fk)

# Summary by site
_by_site = defaultdict(lambda: {"total": 0, "done": 0})
for fk in _file_keys:
    site = fk.split('/')[0]
    _by_site[site]['total'] += 1
for fk in already_done:
    site = fk.split('/')[0]
    _by_site[site]['done'] += 1

print(f'Results folder    : {DRIVE_RESULTS_DIR}')
print(f'Model name        : {MODEL_NAME}')
print(f'DB format         : {DB_FORMAT}')
print(f'Total recordings  : {len(_file_keys)}')
print(f'Already analyzed  : {len(already_done)}')
print(f'Remaining to run  : {len(to_process)}')
print()
for site_name, counts in sorted(_by_site.items()):
    print(f"  {site_name}: {counts['done']}/{counts['total']} done")

if already_done:
    print()
    print('Already done (will be skipped):')
    for fk in already_done[:8]:
        print(f'  {fk}')
    if len(already_done) > 8:
        print(f'  ... and {len(already_done) - 8} more')

if not to_process:
    print()
    print('All recordings have already been classified. Nothing to do!')
    print('Delete the corresponding result files to re-classify.')
else:
    print()
    print(f'Ready to classify {len(to_process)} recording(s). Proceed to Step 5.')

---
## Step 5 — Run Classification & Save Results

For each recording in the embeddings database:
1. All pre-computed embedding vectors for that recording are loaded
2. Each embedding is passed through the classifier head model
3. Detections above the confidence threshold are kept (up to Top K per segment)
4. Results are saved immediately to your Google Drive

**Result files** are saved as:
```
RESULTS_FOLDER / SITE_NAME / SITE_NAME_YYYYMMDD_HHMMSS.MODEL_NAME.results.txt
```

Columns: `site`, `lat`, `lon`, `date`, `time`, `start_time`, `end_time`, `label`, `confidence`

> Depending on the number of recordings and segments, this step may take a while. Progress is shown below.

In [ ]:
#@title 🚀 Run classification { display-mode: "form" }

import csv
import os
import re
import sqlite3
import time
import numpy as np
from datetime import datetime, timedelta

def _parse_stem_datetime(stem, prefix=""):
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})", stem)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf"{re.escape(prefix)}_(\d{{8}})_(\d{{6}})", stem)
    else:
        m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def _apply_activation(logits, activation):
    arr = np.array(logits, dtype=np.float32).flatten()
    if activation == 'sigmoid':
        return 1.0 / (1.0 + np.exp(-arr))
    elif activation == 'softmax':
        arr = arr - arr.max()
        e = np.exp(arr)
        return e / e.sum()
    return arr

def _run_classifier(emb):
    if MODEL_TYPE == 'tflite':
        in_det  = _clf_model.get_input_details()[0]
        out_det = _clf_model.get_output_details()[0]
        _clf_model.resize_tensor_input(in_det['index'], [1, len(emb)])
        _clf_model.allocate_tensors()
        _clf_model.set_tensor(in_det['index'], emb[np.newaxis, :])
        _clf_model.invoke()
        return _clf_model.get_tensor(out_det['index']).flatten()
    else:
        return _clf_model.run(
            [ONNX_OUTPUT_NAME],
            {ONNX_INPUT_NAME: emb[np.newaxis, :].astype(np.float32)},
        )[0].flatten()

def _get_result_path(site, stem):
    _utc_dt  = _parse_stem_datetime(stem, FILENAME_PREFIX)
    _utc_off = _sites_meta.get(site, {}).get('utc_offset', 0.0)
    dt       = _utc_dt + timedelta(hours=_utc_off) if _utc_dt else None
    if dt is not None:
        result_fn = f"{site}_{dt.strftime('%Y%m%d_%H%M%S')}.{MODEL_NAME}.results.txt"
    else:
        result_fn = f'{site}_{stem}.{MODEL_NAME}.results.txt'
    return os.path.join(DRIVE_RESULTS_DIR, site, result_fn)

if not to_process:
    print('No recordings to classify. All analyses are already complete.')
else:
    print(f'Classifying {len(to_process)} recording(s) with model "{MODEL_NAME}"')
    print(f'  Confidence threshold : {MIN_CONFIDENCE}')
    print(f'  Top K                : {TOP_K}')
    print(f'  Window size          : {DB_WINDOW_S} s')
    print(f'  Backbone model       : {DB_MODEL_NAME}')
    print(f'  DB format            : {DB_FORMAT}')
    print('=' * 65)

    total_detections = 0
    total_start      = time.time()

    # Pre-compute stride for the "new" auricularia format (chunk_index → start_time).
    _stride_samples = max(1, int(DB_WINDOW_S * DB_SAMPLE_RATE * (1.0 - DB_OVERLAP)))

    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
        for rec_idx, fk in enumerate(to_process, 1):
            site, stem      = fk.split('/', 1)
            _utc_dt        = _parse_stem_datetime(stem, FILENAME_PREFIX)
            _sm            = _sites_meta.get(site, {})
            _utc_off       = _sm.get('utc_offset', 0.0)
            stream_id      = _sm.get('stream_id', '')
            utc_offset_str = f'UTC{_utc_off:+.1f}'
            file_dt        = _utc_dt + timedelta(hours=_utc_off) if _utc_dt else None
            site_lat       = _sm.get('lat', '') or str(_latlon_map.get(site, ('',''))[0])
            site_lon       = _sm.get('lon', '') or str(_latlon_map.get(site, ('',''))[1])

            print(f'[{rec_idx}/{len(to_process)}] {stem}  (site: {site})')

            if DB_FORMAT == "new":
                _emb_rows = _con.execute(
                    "SELECT chunk_index, data FROM embeddings "
                    "WHERE site_name = ? AND recording_id = ? ORDER BY chunk_index",
                    (site, stem)
                ).fetchall()
            else:
                _emb_rows = _con.execute(
                    "SELECT key, data FROM embeddings WHERE key LIKE ? ORDER BY key",
                    (fk + "/%",)
                ).fetchall()

            rows       = []
            file_start = time.time()

            for row_key, data in _emb_rows:
                if DB_FORMAT == "new":
                    start_s = int(row_key) * _stride_samples / DB_SAMPLE_RATE
                else:
                    start_s = float(row_key.split('/')[2])
                end_s = start_s + DB_WINDOW_S
                emb   = np.frombuffer(data, dtype=np.float32).copy()

                try:
                    logits = _run_classifier(emb)
                except Exception as e:
                    print(f'  ERROR at {start_s:.1f}s: {e}')
                    continue

                scores = _apply_activation(logits, ACTIVATION)

                _seg_hits = sorted(
                    [(float(scores[i]), i)
                     for i in range(min(len(scores), len(labels)))
                     if float(scores[i]) >= MIN_CONFIDENCE],
                    reverse=True
                )[:TOP_K]

                for score, idx in _seg_hits:
                    date_str = file_dt.strftime("%Y-%m-%d") if file_dt else ""
                    time_str = file_dt.strftime("%H:%M:%S") if file_dt else ""
                    rows.append([
                        stream_id, site, site_lat, site_lon,
                        date_str, time_str, utc_offset_str,
                        f"{start_s:.1f}", f"{end_s:.1f}",
                        labels[idx], f"{float(score):.4f}"
                    ])

            result_path = _get_result_path(site, stem)
            os.makedirs(os.path.dirname(result_path), exist_ok=True)
            with open(result_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['stream_id', 'site', 'lat', 'lon', 'date', 'time', 'utc_offset', 'start_time', 'end_time', 'label', 'confidence'])
                writer.writerows(rows)

            elapsed  = time.time() - file_start
            n_segs   = len(_emb_rows)
            per_ms   = elapsed / n_segs * 1000 if n_segs else 0
            total_detections += len(rows)
            print(f'  → {len(rows)} detection(s)  |  {n_segs} segments  |  '
                  f'{elapsed:.1f}s ({per_ms:.0f}ms/seg)  →  {os.path.basename(result_path)}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print('Classification complete!')
    print(f'  Recordings classified : {len(to_process)}')
    print(f'  Total detections      : {total_detections}')
    print(f'  Total time            : {total_elapsed:.1f}s')
    print(f'  Results saved to      : {DRIVE_RESULTS_DIR}')